## Importing libraries
---------------

In [ ]:
import pandas as pd
import pathlib
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from functools import reduce

## Reading Files
---------------

In [ ]:
# Finding CABRa attributes csv files
path_files = pathlib.Path('CABRa_attributes_csv/')
cabra_attributes_files = path_files.rglob('*.csv')

In [ ]:
# Reading files
dfs = []
for file in cabra_attributes_files:
    print(file)
    df = pd.read_csv(file, engine='python', delimiter=';')
    dfs.append(df)

## Creating Dataset and preparing for PCA
-----------------------

In [ ]:
# Merging DataFrames
df_merged = reduce(lambda  left,right: pd.merge(left, right, on=['CABra ID', 'ANA ID'], how='outer'), dfs)

In [ ]:
df_merged.head()

In [ ]:
# All CABRa attributes files
df_merged.columns

In [ ]:
# The column 'catch_area' is problematic, because it was a object type.
# Changed to 'int'
df_merged['catch_area'] = df_merged['catch_area'].str.replace('.','')
df_merged['catch_area'] = df_merged['catch_area'].astype('int64')
df_merged['catch_area'] = df_merged['catch_area']/1000

In [ ]:
# Pre-selection of the variables from CABra columns
variables = ['dist_coast','aridity_index','p_seasonality',  'aquif_name', 'aquif_type',
             'catch_wtd', 'catch_hand', 'hand_class','res_area',  'hdisturb_index',
             'q_elasticity', 'baseflow_index',  'gauge_hreg', 'gauge_biome', 'gauge_state',
             'catch_lith', 'sub_porosity', 'sub_permeability', 'sub_hconduc', 'soil_type',
             'soil_textclass','elev_mean', 'catch_slope', 'catch_order']

In [ ]:
# Removing non-numerical variables
number_variables = []
for v, i in zip(variables, df_merged[variables].dtypes):
    if ('float' in str(i)) or ('int' in str(i)):
        number_variables.append(v)
    else:
        pass

In [ ]:
number_variables

In [ ]:
# Scaling data before PCA
scaler = StandardScaler()
scaler.fit(df_merged[number_variables])
X = scaler.transform(df_merged[number_variables])

In [ ]:
# PCA
pca = PCA(n_components=len(number_variables))

X_new = pca.fit_transform(X)

In [ ]:
# Acumulative variance ratio from PCA
pca.explained_variance_ratio_.cumsum()

In [ ]:
pca.singular_values_

In [ ]:
n_pcs = pca.components_.shape[0]
most_important = [np.abs(pca.components_[i]).argmax() for i in range(n_pcs)]
value = [np.abs(pca.components_[i]).max() for i in range(n_pcs)]

In [ ]:
# Checking top 5 variables from each PC
for j in range(n_pcs):
    print(f'PCA #{j}\t | {pca.explained_variance_ratio_[j]:.2f}')
    for i in np.argsort(np.abs(pca.components_[j]))[-5:]:
        print(f'{number_variables[i]}\t: {pca.components_[j][i]:.2f}\t|{pca.components_[j][i]*pca.explained_variance_ratio_[j]:.2f}')
    print()

In [ ]:
most_important_names = [number_variables[most_important[i]] for i in range(n_pcs)]

In [ ]:
for i in zip(most_important_names, value,pca.explained_variance_ratio_):
    print(i[0], f'|{i[2]:.2f}',f'|{i[1]:.2f}')